# EfficientNetV2 — Smaller Models and Faster Training

**Paper:** Tan, M., & Le, Q. V. (2021). EfficientNetV2: Smaller Models and Faster Training. *ICML 2021*. https://arxiv.org/abs/2104.00298

**EfficientNetV1:** Tan, M., & Le, Q. V. (2019). EfficientNet: Rethinking Model Scaling for CNNs. *ICML 2019*. https://arxiv.org/abs/1905.11946

**Implementation:** `hale_core/nn/vision/cnn/efficientnet.py`

> EfficientNetV2 replaces the initial MBConv stages with FusedMBConv blocks — fusing the expand and depthwise operations into a single 3×3 conv — achieving significantly faster training while maintaining high accuracy.

## Architecture Overview

```
Input [B, 3, H, W]
  → Stem: 3×3 stride-2 conv + BN + SiLU         [B, 24, H/2, W/2]

EfficientNetV2-S Stage Structure:
  Stage 0: FusedMBConv, r=1, k=3, stride=1, C=24,  n=2  → [B,  24, H/2, W/2]
  Stage 1: FusedMBConv, r=4, k=3, stride=2, C=48,  n=4  → [B,  48, H/4, W/4]
  Stage 2: FusedMBConv, r=4, k=3, stride=2, C=64,  n=4  → [B,  64, H/8, W/8]
  Stage 3: MBConv,      r=4, k=3, stride=2, C=128, n=6  → [B, 128, H/16,W/16]
  Stage 4: MBConv,      r=6, k=3, stride=1, C=160, n=9  → [B, 160, H/16,W/16]
  Stage 5: MBConv,      r=6, k=3, stride=2, C=256, n=15 → [B, 256, H/32,W/32]
  Head:    1×1 conv + BN + SiLU                         → [B,1280, H/32,W/32]
  GlobalAvgPool → [B, 1280]

FusedMBConv (early stages — fused expand+depthwise):
  x → 3×3 Conv (expand) → BN → SiLU
    → [SE block if se_ratio]
    → 1×1 Conv (project) → BN
    → (+residual if same dim) via StochasticDepth

MBConv (late stages — factored expand+depthwise):
  x → 1×1 Conv (expand) → BN → SiLU
    → DW 3×3 Conv → BN → SiLU
    → SE block
    → 1×1 Conv (project) → BN
    → (+residual if same dim) via StochasticDepth

Squeeze-Excite (SE):
  x → GlobalAvgPool → fc1 (C//4) → SiLU → fc2 (C) → Sigmoid → scale x
```

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(19, 11))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.axis('off')

def box(ax, x, y, w, h, text, sub='', fc='#1f6feb', fs=8.5):
    r = FancyBboxPatch((x,y), w, h, boxstyle='round,pad=0.07',
                        facecolor=fc, edgecolor='#58a6ff', lw=1.3)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2+(0.08 if sub else 0), text, ha='center', va='center',
            color='white', fontsize=fs, fontweight='bold', fontfamily='monospace')
    if sub:
        ax.text(x+w/2, y+h/2-0.17, sub, ha='center', va='center',
                color='#8b949e', fontsize=6.5)

def arr(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color='#58a6ff', lw=1.4))

# ── LEFT: FusedMBConv ────────────────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 9)
ax.set_ylim(0, 13)
ax.text(4.5, 12.5, 'FusedMBConv', ha='center', color='#f0f6fc',
        fontsize=12, fontweight='bold')
ax.text(4.5, 12.0, '(Early stages 0–2)', ha='center', color='#58a6ff', fontsize=9)

box(ax, 2, 11.0, 5, 0.7, 'Input x', '[B, C_in, H, W]', fc='#21262d')
box(ax, 2, 9.8,  5, 0.8, '3×3 Conv (expand)', 'C_in → C_in×r (fused!)', fc='#1f6feb')
arr(ax, 4.5, 11.0, 4.5, 10.6)
box(ax, 2, 8.8,  5, 0.7, 'BN → SiLU', '', fc='#388bfd')
arr(ax, 4.5, 9.8, 4.5, 9.5)
box(ax, 2, 7.5,  5, 0.9, 'SE Block (optional)', 'GlobalAvgPool→fc1→SiLU→fc2→sigmoid\n→ scale features', fc='#9e6a03', fs=7.5)
arr(ax, 4.5, 8.8, 4.5, 8.4)
box(ax, 2, 6.5,  5, 0.7, '1×1 Conv (project)', 'C_in×r → C_out', fc='#1f6feb')
arr(ax, 4.5, 7.5, 4.5, 7.2)
box(ax, 2, 5.5,  5, 0.7, 'BN (no activation)', '', fc='#388bfd')
arr(ax, 4.5, 6.5, 4.5, 6.2)

# skip
box(ax, 0.2, 8.2, 1.5, 0.7, 'skip\n(if same)', '', fc='#2ea043', fs=7.5)
ax.annotate('', xy=(1.7, 8.5), xytext=(4.5, 11.0),
            arrowprops=dict(arrowstyle='->', color='#f85149', lw=1.5,
                            connectionstyle='arc3,rad=0.3'))

box(ax, 2, 4.5,  5, 0.7, 'StochasticDepth', '', fc='#30363d')
arr(ax, 4.5, 5.5, 4.5, 5.2)
box(ax, 2, 3.5,  5, 0.7, 'Add (+skip)', '', fc='#2ea043')
arr(ax, 4.5, 4.5, 4.5, 4.2)
ax.annotate('', xy=(3.0, 3.7), xytext=(1.7, 8.2),
            arrowprops=dict(arrowstyle='->', color='#f85149', lw=1.5))
box(ax, 2, 2.5,  5, 0.7, 'Output', '[B, C_out, H\', W\']', fc='#21262d')
arr(ax, 4.5, 3.5, 4.5, 3.2)

ax.text(4.5, 2.0, 'Fusing expand + DW into single 3×3', ha='center', color='#8b949e', fontsize=8)
ax.text(4.5, 1.6, 'reduces memory I/O on large feature maps', ha='center', color='#8b949e', fontsize=8)

# ── MIDDLE: MBConv ────────────────────────────────────────────────────────────
ax = axes[1]
ax.set_xlim(0, 9)
ax.set_ylim(0, 13)
ax.text(4.5, 12.5, 'MBConv', ha='center', color='#f0f6fc',
        fontsize=12, fontweight='bold')
ax.text(4.5, 12.0, '(Late stages 3–5)', ha='center', color='#58a6ff', fontsize=9)

box(ax, 2, 11.0, 5, 0.7, 'Input x', '[B, C_in, H, W]', fc='#21262d')
box(ax, 2, 9.8,  5, 0.8, '1×1 Conv (expand)', 'C_in → C_in×r', fc='#1f6feb')
arr(ax, 4.5, 11.0, 4.5, 10.6)
box(ax, 2, 8.8,  5, 0.7, 'BN → SiLU', '', fc='#388bfd')
arr(ax, 4.5, 9.8, 4.5, 9.5)
box(ax, 2, 7.8,  5, 0.7, 'DW 3×3 Conv', 'groups=C_in×r', fc='#1f6feb')
arr(ax, 4.5, 8.8, 4.5, 8.5)
box(ax, 2, 6.8,  5, 0.7, 'BN → SiLU', '', fc='#388bfd')
arr(ax, 4.5, 7.8, 4.5, 7.5)
box(ax, 2, 5.6,  5, 0.9, 'SE Block', 'GlobalAvgPool→fc1→SiLU→fc2→sigmoid\nse_ratio=0.25', fc='#9e6a03', fs=7.5)
arr(ax, 4.5, 6.8, 4.5, 6.5)
box(ax, 2, 4.6,  5, 0.7, '1×1 Conv (project)', 'C_in×r → C_out', fc='#1f6feb')
arr(ax, 4.5, 5.6, 4.5, 5.3)
box(ax, 2, 3.6,  5, 0.7, 'BN (no activation)', '', fc='#388bfd')
arr(ax, 4.5, 4.6, 4.5, 4.3)
box(ax, 2, 2.6,  5, 0.7, 'StochasticDepth + Add', '', fc='#2ea043')
arr(ax, 4.5, 3.6, 4.5, 3.3)
ax.annotate('', xy=(3.0, 2.7), xytext=(1.2, 11.0),
            arrowprops=dict(arrowstyle='->', color='#f85149', lw=1.5,
                            connectionstyle='arc3,rad=0.0'))
box(ax, 2, 1.7,  5, 0.7, 'Output', '[B, C_out, H\', W\']', fc='#21262d')
arr(ax, 4.5, 2.6, 4.5, 2.4)

ax.text(4.5, 1.2, 'Factored DW separates spatial+channel mixing', ha='center', color='#8b949e', fontsize=8)
ax.text(4.5, 0.8, 'efficient at small spatial sizes (late stages)', ha='center', color='#8b949e', fontsize=8)

# ── RIGHT: SE + stage mapping ────────────────────────────────────────────────
ax = axes[2]
ax.set_xlim(0, 9)
ax.set_ylim(0, 13)
ax.text(4.5, 12.5, 'SqueezeExcite + Stage Map', ha='center',
        color='#f0f6fc', fontsize=11, fontweight='bold')

box(ax, 2, 11.2, 5, 0.7, 'Feature map x', '[B, C, H, W]', fc='#21262d')
box(ax, 2, 10.1, 5, 0.7, 'GlobalAvgPool', '→ [B, C, 1, 1]', fc='#1f6feb')
arr(ax, 4.5, 11.2, 4.5, 10.8)
box(ax, 2, 9.0,  5, 0.7, 'fc1: C → C//4', '→ SiLU', fc='#388bfd')
arr(ax, 4.5, 10.1, 4.5, 9.7)
box(ax, 2, 7.9,  5, 0.7, 'fc2: C//4 → C', '→ Sigmoid → [0,1]', fc='#388bfd')
arr(ax, 4.5, 9.0, 4.5, 8.6)
box(ax, 2, 6.9,  5, 0.7, 'Scale: x × se_out', 'channel-wise multiply', fc='#2ea043')
arr(ax, 4.5, 7.9, 4.5, 7.6)
arr(ax, 4.5, 6.9, 4.5, 6.6)
box(ax, 2, 5.9,  5, 0.7, 'Recalibrated Output', '', fc='#21262d')

# stage table
ax.text(4.5, 5.2, 'Stage   Type        r   C    n', ha='center',
        color='#58a6ff', fontsize=8, fontfamily='monospace')
rows = [
    ('0', 'FusedMBConv', '1', '24',  '2',  '#6e40c9'),
    ('1', 'FusedMBConv', '4', '48',  '4',  '#6e40c9'),
    ('2', 'FusedMBConv', '4', '64',  '4',  '#6e40c9'),
    ('3', 'MBConv',      '4', '128', '6',  '#1f6feb'),
    ('4', 'MBConv',      '6', '160', '9',  '#1f6feb'),
    ('5', 'MBConv',      '6', '256', '15', '#1f6feb'),
]
for i, (s, t, r, c, n, color) in enumerate(rows):
    ax.text(4.5, 4.7 - i*0.5, f'{s:3}   {t:12}  {r:3} {c:4} {n:3}',
            ha='center', color=color, fontsize=8, fontfamily='monospace')

ax.text(4.5, 1.7, 'Progressive learning rate (EfficientNetV2):', ha='center',
        color='#f0f6fc', fontsize=8.5, fontweight='bold')
ax.text(4.5, 1.2, 'Train with small images first, progressively increase,', ha='center',
        color='#8b949e', fontsize=8)
ax.text(4.5, 0.8, 'also increase regularisation (dropout, mixup, RandAugment)', ha='center',
        color='#8b949e', fontsize=8)

plt.suptitle('EfficientNetV2 Architecture', fontsize=14, color='#f0f6fc',
             fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()


## Key Innovations

### 1. The Training Speed Problem
EfficientNetV1 (2019) used NAS to find the optimal combination of depth/width/resolution scaling. It achieved state-of-the-art accuracy. However, it had two training bottlenecks:
- **MBConv with large depthwise convolutions at early stages** (large feature maps → memory bandwidth bottleneck)
- **Very large image sizes** in progressive training caused too much memory pressure

### 2. FusedMBConv vs MBConv
**MBConv** (from MobileNetV2, 2018):
```
1×1 expand → DW 3×3 → 1×1 project
```
At early stages, the feature map is *large* and channels are *few*. The expand 1×1 makes the channel dimension large, then DW processes spatially. This causes massive memory reads on large spatial maps.

**FusedMBConv** merges the first two operations:
```
3×3 expand+conv → 1×1 project
```
A single 3×3 conv handles both the expansion and spatial mixing. This is more cache-friendly on modern accelerators (fewer memory transactions) while achieving similar accuracy.

### 3. Progressive Learning Schedule
Train with progressively larger images: start with small images and low regularisation, end with large images and strong regularisation. This reduces the total training time while maintaining accuracy comparable to training at full size throughout.

### 4. Squeeze-Excite (SE) Channel Attention
SE computes a global channel-wise importance vector from a squeeze (GlobalAvgPool) followed by two fc layers. The result is applied to scale each channel's feature map. This recalibrates which channels are useful for the current input — a lightweight form of channel self-attention.

### 5. Stochastic Depth (DropPath)
Randomly drop entire blocks during training (survival probability). This acts as a regulariser that prevents over-fitting and improves generalisation, especially for deep networks. Rates are linearly scaled from 0 (first block) to `drop_path_rate` (last block).

### 6. SiLU (Swish) Activation
SiLU(x) = x · sigmoid(x) — smooth, non-monotonic activation. Empirically outperforms ReLU in EfficientNet architectures due to its non-linearity for negative values.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import torch
from hale_core.nn.vision import build_vision_backbone

x = torch.randn(1, 3, 224, 224)

for name in ['efficientnet_v2_s', 'efficientnet_v2_m', 'efficientnet_v2_l']:
    bb = build_vision_backbone(name, img_size=224)
    bb.eval()
    with torch.no_grad():
        out = bb(x)
    print(f'{name:22s}  embed={out.pooled_output.shape[-1]}')
    print(f'  feature_maps: {[tuple(f.shape) for f in out.feature_maps]}')
    print(f'  params: {bb.num_parameters():,}')
    print()


## References

1. **[Primary]** Tan, M., & Le, Q. V. (2021). **EfficientNetV2: Smaller Models and Faster Training**. *ICML 2021*. https://arxiv.org/abs/2104.00298

2. Tan, M., & Le, Q. V. (2019). **EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks**. *ICML 2019*. https://arxiv.org/abs/1905.11946 — The V1 that V2 builds on.

3. Sandler, M., Howard, A., Zhu, M., Zhmoginov, A., & Chen, L.-C. (2018). **MobileNetV2: Inverted Residuals and Linear Bottlenecks**. *CVPR 2018*. — The MBConv block used in EfficientNet.

4. Hu, J., Shen, L., & Sun, G. (2018). **Squeeze-and-Excitation Networks**. *CVPR 2018*. — The SE block used in EfficientNet for channel attention.

5. Huang, G., Sun, Y., Liu, Z., Sedra, D., & Weinberger, K. (2016). **Deep Networks with Stochastic Depth**. *ECCV 2016*. — Stochastic depth (DropPath) regularisation.